# 02_silver_transform — Cleaning and Standardisation
## ClinVar Medallion Pipeline · Silver Layer

Transforms raw Bronze data into a clean, validated Silver table.

**What happens here:**
- Filter to the GRCh38 assembly only — removes GRCh37 duplicate build
- Replace NCBI missing-value markers (`-`, `na`) with proper nulls
- Standardise `ClinicalSignificance` into 7 canonical categories
- Cast types: `LastEvaluated` → date (via a Python UDF), `GeneSymbol` → uppercase
- Add `dq_flag` / `dq_reason` — bad rows are flagged, never dropped

**Design principle:** Silver enforces quality without losing auditability.
Every row that fails a quality check is kept with a flag explaining why —
silent drops would make problems invisible, which is unacceptable in a
regulated/clinical context.

**Run `00_setup` and `01_bronze_ingestion` first.**

In [0]:
%run ./00_setup

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# READ FROM BRONZE DELTA TABLE
# ─────────────────────────────────────────────────────────────────────────────
# Silver reads the Bronze *table*, never the raw file — only Bronze touches
# the source. Re-running Bronze means Silver picks up changes on its next run.

print(f"Reading from Bronze table: {TABLE_BRONZE}")

df = spark.table(TABLE_BRONZE)

print(f"Rows in Bronze   : {df.count():,}")
print(f"Columns          : {len(df.columns)}")
print("\nReady for Silver transformations.")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# FILTER TO GRCh38 ASSEMBLY ONLY
# ─────────────────────────────────────────────────────────────────────────────
# Each variant appears once per genome build (GRCh37 + GRCh38). Keeping both
# would double every Gold count. Drop GRCh37/na/NCBI36, keep current standard.
# Uses the GENOME_ASSEMBLY constant from setup (one place to change).

rows_before = df.count()

df = df.filter(df.Assembly == GENOME_ASSEMBLY)

rows_after = df.count()
rows_dropped = rows_before - rows_after

print(f"Assembly filter applied:")
print(f"  Rows before : {rows_before:,}")
print(f"  Rows after  : {rows_after:,}")
print(f"  Rows dropped: {rows_dropped:,}  ({rows_dropped/rows_before*100:.1f}%)")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# REPLACE NCBI MISSING VALUE MARKERS WITH PROPER NULLS
# ─────────────────────────────────────────────────────────────────────────────
# ClinVar marks "missing" with text ('-', 'na'), so Bronze showed 0% nulls —
# the gaps were hidden as strings. Convert them to real nulls so downstream
# isNull() checks work. String columns only; trim() catches padded markers.

from pyspark.sql.functions import col, when, trim
from pyspark.sql import functions as F

# Identify string columns
string_cols = [f.name for f in df.schema.fields 
               if str(f.dataType) == "StringType()"]

print(f"String columns to clean: {len(string_cols)}")

# Replace '-' and 'na' with null across all string columns
# trim() handles any accidental whitespace around the marker
for c in string_cols:
    df = df.withColumn(
        c,
        when(trim(col(c)).isin("-", "na", "NA", ""), None).otherwise(col(c))
    )

print("Missing value markers replaced with null.")

print("\nNull counts after replacement (key columns):")
key_cols = ["GeneSymbol", "ClinicalSignificance", "PhenotypeList",
            "Assembly", "Chromosome", "LastEvaluated"]

null_counts = df.select([
    F.sum(col(c).isNull().cast("int")).alias(c)
    for c in key_cols
]).collect()[0]

for c in key_cols:
    pct = (null_counts[c] / rows_after) * 100
    print(f"  {c:35s}: {null_counts[c]:>7,}  ({pct:.1f}%)")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STANDARDISE CLINICALSIGNIFICANCE
# ─────────────────────────────────────────────────────────────────────────────
# Raw ClinicalSignificance is a free-text field with inconsistent values.
# We map everything to the 7 canonical categories defined in 00_setup.
#
# Strategy:
# 1. Lowercase and trim for consistent matching
# 2. Extract the first meaningful term (before / or ;)
# 3. Map to canonical category via a priority-ordered keyword match
# 4. Anything unmatched → "Other / Not provided"
#
# A new column 'ClinicalSignificance_clean' is added.
# The original 'ClinicalSignificance' column is preserved — Bronze fidelity
# principle extends into Silver for this column specifically.

from pyspark.sql.functions import (
    col, when, lower, trim, split, regexp_replace
)

def standardise_significance(c):
    """
    Takes the raw ClinicalSignificance column and returns a cleaned version.
    Applied as a PySpark column expression, not a Python UDF —
    column expressions execute on executors; UDFs have serialisation overhead.
    """
    # Step 1: lowercase and trim
    cleaned = trim(lower(col(c)))

    # Step 2: take only the part before the first '/' or ';'
    # This handles conflations like "Likely pathogenic/Pathogenic"
    # and modifiers like "Uncertain significance; risk factor"
    cleaned = trim(split(cleaned, "[/;]")[0])

    # Step 3: map to canonical categories via keyword matching
    # Order matters — more specific patterns before more general ones
    return (
        when(cleaned.isNull(), None)
        .when(cleaned.contains("pathogenic, low penetrance"), "Other / Not provided")
        .when(cleaned.contains("risk allele"), "Other / Not provided")
        .when(cleaned.contains("likely pathogenic"), "Likely pathogenic")
        .when(cleaned.contains("pathogenic"), "Pathogenic")
        .when(cleaned.contains("uncertain significance"), "Uncertain significance")
        .when(cleaned.contains("likely benign"), "Likely benign")
        .when(cleaned.contains("benign"), "Benign")
        .when(cleaned.contains("conflicting"), "Conflicting interpretations of pathogenicity")
        .when(cleaned.contains("drug response"), "Other / Not provided")
        .when(cleaned.contains("association"), "Other / Not provided")
        .when(cleaned.contains("risk factor"), "Other / Not provided")
        .when(cleaned.contains("protective"), "Other / Not provided")
        .when(cleaned.contains("affects"), "Other / Not provided")
        .when(cleaned.contains("other"), "Other / Not provided")
        .when(cleaned.contains("not provided"), "Other / Not provided")
        .when(cleaned.contains("not classified"), "Other / Not provided")
        .otherwise("Other / Not provided")
    )

df = df.withColumn(
    "ClinicalSignificance_clean",
    standardise_significance("ClinicalSignificance")
)

# Show distribution of clean values
print("ClinicalSignificance_clean distribution:")
(
    df.groupBy("ClinicalSignificance_clean")
    .count()
    .orderBy("count", ascending=False)
    .show(truncate=False)
)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# TYPE CASTING AND COLUMN STANDARDISATION
# ─────────────────────────────────────────────────────────────────────────────
# GeneSymbol → uppercase so the same gene doesn't split across casings in Gold.
# LastEvaluated → date via a Python UDF: Serverless locks Spark's date parser
# (timeParserPolicy), so strptime in plain Python sidesteps that lock.
# Unparseable/null dates return None → caught by dq_flag next.

from pyspark.sql.functions import upper, udf
from pyspark.sql.types import DateType
from datetime import datetime

# Python UDF — runs Python on each row, bypasses Spark's date parser entirely
# Slower than a column expression but immune to Spark date config issues

def parse_date(date_str):
    """
    Tries to parse the date string. Returns None if it fails.
    None in a UDF becomes null in the Delta table.
    """
    if date_str is None:
        return None
    try:
        return datetime.strptime(date_str.strip(), "%b %d, %Y").date()
    except:
        return None

parse_date_udf = udf(parse_date, DateType())

# GeneSymbol — uppercase
df = df.withColumn(
    "GeneSymbol",
    upper(col("GeneSymbol"))
)

non_null = df.filter(col("LastEvaluated").isNotNull()).count()
total = df.count()
print(f"  LastEvaluated non-null: {non_null:,} of {total:,}")

# LastEvaluated — parsed via UDF
df = df.withColumn(
    "LastEvaluated",
    parse_date_udf(col("LastEvaluated"))
)

print("Type casting applied:")
print("  LastEvaluated → date (via Python UDF)")
print("  GeneSymbol    → uppercase\n")

print("Sample rows (key columns):")
(
    df.select(
        "VariationID",
        "GeneSymbol",
        "ClinicalSignificance_clean",
        "LastEvaluated",
        "Assembly"
    )
    .filter(col("GeneSymbol").isNotNull())
    .limit(5)
    .show(truncate=False)
)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA QUALITY FLAGS
# ─────────────────────────────────────────────────────────────────────────────
# dq_flag = True if any key field is missing; dq_reason gives the first cause.
# Bad rows are kept, not deleted, so the data stays auditable — analysts filter
# dq_flag = False for clean analysis or investigate flagged rows separately.
# (Compliance argument: silent drops mean you can't prove what happened.)

from pyspark.sql.functions import col, when

df = df.withColumn(
    "dq_flag",
    when(col("ClinicalSignificance_clean").isNull(), True)
    .when(col("GeneSymbol").isNull(), True)
    .when(col("PhenotypeList").isNull(), True)
    .when(col("LastEvaluated").isNull(), True)
    .otherwise(False)
).withColumn(
    "dq_reason",
    when(col("ClinicalSignificance_clean").isNull(), "Missing clinical significance")
    .when(col("GeneSymbol").isNull(), "Missing gene symbol")
    .when(col("PhenotypeList").isNull(), "Missing phenotype")
    .when(col("LastEvaluated").isNull(), "Missing or unparseable evaluation date")
    .otherwise(None)
)

# Summary
print("Data quality flag summary:")
(
    df.groupBy("dq_flag", "dq_reason")
    .count()
    .orderBy("dq_flag", "count", ascending=[True, False])
    .show(truncate=False)
)

total = df.count()
flagged = df.filter(col("dq_flag") == True).count()
clean = total - flagged

print(f"\nTotal rows   : {total:,}")
print(f"Clean rows   : {clean:,}  ({clean/total*100:.1f}%)")
print(f"Flagged rows : {flagged:,}  ({flagged/total*100:.1f}%)")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# SELECT FINAL COLUMNS AND WRITE SILVER DELTA TABLE
# ─────────────────────────────────────────────────────────────────────────────
# Keep only the columns Gold needs (44 → 19); the rest stay in Bronze.
# Keep BOTH raw and clean significance side by side for traceability.
# DROP first so the new schema applies cleanly after logic changes.

from pyspark.sql.functions import current_timestamp

# Drop the Silver table if it exists from a previous failed attempt
spark.sql(f"DROP TABLE IF EXISTS {TABLE_SILVER}")
print(f"Dropped existing table: {TABLE_SILVER}")

df_silver = df.select(
    "VariationID",
    "AlleleID",
    "GeneSymbol",
    "ClinicalSignificance",           # original — preserved for traceability
    "ClinicalSignificance_clean",     # standardised — used in Gold
    "PhenotypeList",
    "ReviewStatus",
    "NumberSubmitters",
    "Chromosome",
    "Start",
    "Stop",
    "Assembly",
    "Type",
    "Origin",
    "LastEvaluated",
    "dq_flag",
    "dq_reason",
    "ingestion_timestamp"
).withColumn("silver_timestamp", current_timestamp())

print(f"Writing to Silver Delta table: {TABLE_SILVER}")
print(f"Columns selected : {len(df_silver.columns)}")
print(f"Rows to write    : {df_silver.count():,}\n")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_SILVER)
)

print(f"Silver table written successfully.")
print(f"Table            : {TABLE_SILVER}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# SILVER VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
# Read back from the Delta table to confirm the write persisted. Checks the
# dq_flag split, previews the clean-only significance distribution (what Gold
# aggregates), and reconciles row counts Bronze → Silver so every row is
# accounted for — nothing dropped silently.

print("=" * 60)
print("SILVER VALIDATION")
print("=" * 60)

df_check = spark.table(TABLE_SILVER)

# Row and column count
print(f"\n  Rows    : {df_check.count():,}")
print(f"  Columns : {len(df_check.columns)}")

# dq_flag distribution
print("\n  Data quality flag distribution:")
(
    df_check
    .groupBy("dq_flag")
    .count()
    .orderBy("dq_flag")
    .show()
)

# ClinicalSignificance_clean distribution — clean rows only
print("  ClinicalSignificance_clean (clean rows only):")
(
    df_check
    .filter(col("dq_flag") == False)
    .groupBy("ClinicalSignificance_clean")
    .count()
    .orderBy("count", ascending=False)
    .show(truncate=False)
)

# Bronze vs Silver row counts
bronze_count = spark.table(TABLE_BRONZE).count()
silver_count = df_check.count()
flagged_count = df_check.filter(col("dq_flag") == True).count()
clean_count = silver_count - flagged_count

print("  Row count reconciliation:")
print(f"    Bronze (all assemblies) : {bronze_count:,}")
print(f"    Silver (GRCh38 only)    : {silver_count:,}")
print(f"    Dropped by assembly     : {bronze_count - silver_count:,}")
print(f"    Silver clean rows       : {clean_count:,}")
print(f"    Silver flagged rows     : {flagged_count:,}")

# Sample clean rows
print("\n  Sample clean rows:")
(
    df_check
    .filter(col("dq_flag") == False)
    .select(
        "VariationID",
        "GeneSymbol",
        "ClinicalSignificance_clean",
        "LastEvaluated",
        "dq_flag"
    )
    .limit(5)
    .show(truncate=False)
)

print("=" * 60)
print("  SILVER COMPLETE — safe to run 03_gold_aggregations")
print("=" * 60)